<a href="https://colab.research.google.com/github/Harsh123-pal/SwBuilder/blob/master/biomatreicverificationusingbiometriclearningnsut7semproject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet tensorflow-federated

In [ ]:
import os
from google.colab import files

print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Set up Kaggle directory and permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and extract the FER-2013 dataset
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d ./data
!tar -xzf ./data/fer2013.tar.gz -C ./data/

In [ ]:
# Force overwrite without prompting
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d ./data

# Extract the nested fer2013 dataset
!tar -xzf ./data/fer2013.tar.gz -C ./data/

# Check that the fer2013.csv file exists
!ls -lh ./data/fer2013

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
csv_path = "./data/fer2013/fer2013.csv"
df = pd.read_csv(csv_path)

# Separate pixels and labels
def process_data(subset_df):
    pixels = subset_df['pixels'].apply(lambda x: np.fromstring(x, sep=' ', dtype=np.float32))
    x = np.stack(pixels.values).reshape(-1, 48, 48, 1) / 255.0
    y = subset_df['emotion'].values.astype(np.int32)
    return x, y

train_df = df[df['Usage'] == 'Training']
test_df = df[df['Usage'] == 'PublicTest']

x_train, y_train = process_data(train_df)
x_test, y_test = process_data(test_df)

# Split training data evenly among 3 clients
num_clients = 3
client_x = np.array_split(x_train, num_clients)
client_y = np.array_split(y_train, num_clients)
client_data = list(zip(client_x, client_y))

print(f"Total training samples: {len(x_train)}")
print(f"Samples per client: {[len(c[0]) for c in client_data]}")
print(f"Test samples: {len(x_test)}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 4-Block VGG-inspired CNN architecture from the repository
def build_client_model():
    model = models.Sequential([
        layers.Input(shape=(48, 48, 1)),

        # Block 1
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classification Head
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(7, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=0.02),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Federated Averaging (FedAvg): element-wise average of client weight tensors
def federate_average_weights(client_weights_list):
    new_weights = []
    for weights_tuple in zip(*client_weights_list):
        new_weights.append(np.array(weights_tuple).mean(axis=0))
    return new_weightsZ

print("Model architecture and FedAvg aggregation ready.")

In [ ]:
import numpy as np

def federate_average_weights(client_weights_list):
    new_weights = []
    for weights_tuple in zip(*client_weights_list):
        new_weights.append(np.array(weights_tuple).mean(axis=0))
    return new_weights  # Fixed: removed trailing 'Z'

print("federate_average_weights corrected.")

In [ ]:
# Initialize the global server model
global_model = build_client_model()

# Training settings
NUM_ROUNDS = 5     # Start with 5 rounds to verify (increase to 15-20 later if you want higher accuracy)
LOCAL_EPOCHS = 1   # Number of epochs each client trains on their private dataset per round
BATCH_SIZE = 64

history = {'round': [], 'loss': [], 'accuracy': []}

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n{'='*20} Starting Federated Round {round_num}/{NUM_ROUNDS} {'='*20}")
    local_weights = []

    # Simulate decentralized client devices
    for client_id in range(num_clients):
        # 1. Download current global weights to client device
        client_model = build_client_model()
        client_model.set_weights(global_model.get_weights())

        # 2. Local client-side training (data never leaves the client)
        c_x, c_y = client_data[client_id]
        print(f"Training on Client {client_id + 1}...")
        client_model.fit(c_x, c_y, epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1)

        # 3. Client sends parameter updates (weights) back to server
        local_weights.append(client_model.get_weights())

    # Central Server: Aggregate client weights via FedAvg
    averaged_weights = federate_average_weights(local_weights)
    global_model.set_weights(averaged_weights)

    # Evaluate global model on unseen test data
    test_loss, test_acc = global_model.evaluate(x_test, y_test, verbose=0)
    history['round'].append(round_num)
    history['loss'].append(test_loss)
    history['accuracy'].append(test_acc)

    print(f">> Round {round_num} Completed | Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc * 100:.2f}%")

print("\nFederated training finished successfully!")

In [ ]:
preds = global_model.predict(x_test[:100], verbose=0)
pred_classes = np.argmax(preds, axis=1)
print("Predicted class distribution in first 100 test samples:")
print(pd.Series(pred_classes).value_counts())


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Clean architecture without BatchNorm interference
def build_stable_fl_model():
    model = models.Sequential([
        layers.Input(shape=(48, 48, 1)),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classification Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(7, activation='softmax')
    ])

    # SGD with momentum works consistently with FedAvg weight averaging
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def federate_average_weights(client_weights_list):
    new_weights = []
    for weights_tuple in zip(*client_weights_list):
        new_weights.append(np.array(weights_tuple).mean(axis=0))
    return new_weights

# 2. Re-train with 2 local epochs per round
global_model = build_stable_fl_model()
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2
BATCH_SIZE = 64

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n--- Federated Round {round_num}/{NUM_ROUNDS} ---")
    local_weights = []

    for client_id in range(num_clients):
        client_model = build_stable_fl_model()
        client_model.set_weights(global_model.get_weights())

        c_x, c_y = client_data[client_id]
        client_model.fit(c_x, c_y, epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        local_weights.append(client_model.get_weights())

    averaged_weights = federate_average_weights(local_weights)
    global_model.set_weights(averaged_weights)

    test_loss, test_acc = global_model.evaluate(x_test, y_test, verbose=0)
    print(f"Round {round_num} -> Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
preds = global_model.predict(x_test[:100], verbose=0)
pred_classes = np.argmax(preds, axis=1)
print(pd.Series(pred_classes).value_counts())

In [ ]:
import matplotlib.pyplot as plt

# FER-2013 7 Emotion categories
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# Pick 5 test samples
sample_images = x_test[:5]
sample_true_labels = y_test[:5]

# Predict
raw_preds = global_model.predict(sample_images, verbose=0)
pred_labels = np.argmax(raw_preds, axis=1)

# Plot
plt.figure(figsize=(14, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(sample_images[i].reshape(48, 48), cmap='gray')
    pred_name = emotion_labels[pred_labels[i]]
    true_name = emotion_labels[sample_true_labels[i]]

    color = 'green' if pred_labels[i] == sample_true_labels[i] else 'red'
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Save in native Keras format
global_model.save("federated_facial_expression_model.keras")

# Trigger download to your local machine
from google.colab import files
files.download("federated_facial_expression_model.keras")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print("Upload an image file (.jpg, .png):")
uploaded = files.upload()

for filename in uploaded.keys():
    # Read image
    img = cv2.imread(filename)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize to the model's expected 48x48 input shape
    resized = cv2.resize(gray, (48, 48), interpolation=cv2.INTER_AREA)
    normalized = resized.astype('float32') / 255.0
    input_tensor = np.expand_dims(normalized, axis=(0, -1)) # Shape: (1, 48, 48, 1)

    # Predict
    preds = global_model.predict(input_tensor, verbose=0)[0]
    predicted_idx = np.argmax(preds)
    predicted_class = emotion_labels[predicted_idx]
    confidence = preds[predicted_idx] * 100

    # Display image and class probabilities
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.set_title(f"Prediction: {predicted_class} ({confidence:.1f}%)")
    ax1.axis('off')

    ax2.barh(emotion_labels, preds, color='skyblue')
    ax2.set_xlim(0, 1.0)
    ax2.set_xlabel('Probability')
    ax2.set_title('Confidence per Emotion')
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

# FER-2013 class names
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

print("Upload an image file (.jpg, .png):")
uploaded = files.upload()

for filename in uploaded.keys():
    # 1. Open image using PIL
    pil_img = Image.open(filename).convert('RGB')
    w, h = pil_img.size

    # 2. Square-crop the center (focuses on face/subject)
    min_dim = min(w, h)
    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    right = left + min_dim
    bottom = top + min_dim
    cropped_img = pil_img.crop((left, top, right, bottom))

    # 3. Convert to 48x48 grayscale for the model
    gray_img = cropped_img.convert('L').resize((48, 48), Image.Resampling.LANCZOS)
    normalized = np.array(gray_img, dtype='float32') / 255.0
    input_tensor = normalized.reshape(1, 48, 48, 1)

    # 4. Predict emotion
    preds = global_model.predict(input_tensor, verbose=0)[0]
    predicted_idx = np.argmax(preds)
    predicted_class = emotion_labels[predicted_idx]
    confidence = preds[predicted_idx] * 100

    # 5. Display the image and emotion probabilities
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.imshow(cropped_img)
    ax1.set_title(f"Prediction: {predicted_class} ({confidence:.1f}%)")
    ax1.axis('off')

    ax2.barh(emotion_labels, preds, color='dodgerblue')
    ax2.set_xlim(0, 1.0)
    ax2.set_xlabel('Probability')
    ax2.set_title('Confidence per Emotion')
    plt.tight_layout()
    plt.show()

In [ ]:
# Save native Keras model
global_model.save("federated_facial_expression_model.keras")

# Download directly to your PC
from google.colab import files
files.download("federated_facial_expression_model.keras")

In [ ]:
# Add simple augmentation to client training batches
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

# Re-run 10 additional rounds starting from current global_model weights
ADDITIONAL_ROUNDS = 10
for round_num in range(11, 11 + ADDITIONAL_ROUNDS):
    print(f"\n--- Federated Round {round_num}/{10 + ADDITIONAL_ROUNDS} ---")
    local_weights = []

    for client_id in range(num_clients):
        client_model = build_stable_fl_model()
        client_model.set_weights(global_model.get_weights())

        c_x, c_y = client_data[client_id]
        # Augment training slice
        c_x_aug = data_augmentation(c_x, training=True).numpy()

        client_model.fit(c_x_aug, c_y, epochs=2, batch_size=64, verbose=0)
        local_weights.append(client_model.get_weights())

    global_model.set_weights(federate_average_weights(local_weights))
    test_loss, test_acc = global_model.evaluate(x_test, y_test, verbose=0)
    print(f"Round {round_num} -> Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc * 100:.2f}%")

In [ ]:
from google.colab import files

# Save native Keras format
global_model.save("federated_facial_expression_model_v2.keras")
print("Saved federated_facial_expression_model_v2.keras")

# Trigger download to your local machine
files.download("federated_facial_expression_model_v2.keras")

In [ ]:
import matplotlib.pyplot as plt

rounds = list(range(1, 21))
# Combined accuracy and loss progression across all 20 rounds
accuracies = [
    24.97, 27.42, 33.38, 36.19, 40.43, 42.94, 45.00, 45.89, 48.15, 50.04,
    49.85, 50.96, 51.60, 52.94, 53.44, 53.91, 54.64, 54.42, 56.00, 56.37
]
losses = [
    1.8027, 1.7638, 1.6907, 1.6187, 1.5394, 1.4894, 1.4288, 1.3936, 1.3444, 1.3036,
    1.2954, 1.2686, 1.2508, 1.2214, 1.2106, 1.2036, 1.1909, 1.1761, 1.1615, 1.1496
]

fig, ax1 = plt.subplots(figsize=(9, 5))

# Plot accuracy
color = 'tab:blue'
ax1.set_xlabel('Federated Communication Round', fontsize=12)
ax1.set_ylabel('Global Test Accuracy (%)', color=color, fontsize=12)
line1 = ax1.plot(rounds, accuracies, color=color, marker='o', linewidth=2, label='Test Accuracy')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

# Plot loss on dual axis
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Global Test Loss', color=color, fontsize=12)
line2 = ax2.plot(rounds, losses, color=color, marker='s', linewidth=2, linestyle='--', label='Test Loss')
ax2.tick_params(axis='y', labelcolor=color)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')

plt.title('Federated Learning Convergence on FER-2013 (20 Rounds)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Run predictions on all 3,589 test images
raw_preds = global_model.predict(x_test, batch_size=128, verbose=0)
y_pred = np.argmax(raw_preds, axis=1)

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_labels,
            yticklabels=emotion_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - 20 Rounds FedAvg')
plt.show()

# Print detailed per-class metrics
print(classification_report(y_test, y_pred, target_names=emotion_labels))

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(global_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("fer_model_v2.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Exported TFLite size: {len(tflite_model) / 1024:.1f} KB")
files.download("fer_model_v2.tflite")

In [ ]:
import numpy as np
import tensorflow as tf

# Load TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="fer_model_v2.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Test with a single sample from test set
test_sample = x_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()

output_data = interpreter.get_tensor(output_details[0]['index'])
predicted_class = emotion_labels[np.argmax(output_data)]
print(f"TFLite inference successful! Predicted class: {predicted_class}")

In [ ]:
from google.colab import files

files.download("federated_facial_expression_model_v2.keras")
files.download("fer_model_v2.tflite")